# Fabric Framework certification Pipeline worker

This notebook is invoked only by the reusable certification Data Pipeline. It receives the exact seven Framework correlation parameters and resolves credentials at runtime.

In [ ]:
# Framework-owned dynamic parameters. Do not rename.
framework_pipeline_run_id = ""
framework_dataset_run_id = ""
dataset_id = ""
run_mode = "NORMAL"
attempt = 1
effective_config_hash = ""
execution_plan_hash = ""

# Environment-local, non-secret deployment settings.
customer_inputs_root = "/lakehouse/default/Files/framework_cert/customer-inputs"
key_vault_url = ""
control_plane_secret_name = ""
warehouse_secret_name = ""

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys

from notebookutils import credentials
from sqlalchemy import create_engine

from fabric_data_framework.control_plane.sqlalchemy_repository import SqlAlchemyControlPlaneRepository
from fabric_data_framework.deployment.contracts import ReleaseManifest
from fabric_data_framework.deployment.delivery import load_dataset_configs
from fabric_data_framework.execution import execute_pipeline_child, pipeline_child_request_from_parameters

root = Path(customer_inputs_root)
inputs = json.loads((root / "INPUTS.json").read_text(encoding="utf-8"))
manifest = ReleaseManifest.model_validate_json((root / "release-manifest.json").read_text(encoding="utf-8"))
wheel_name = inputs["extension_wheel_filename"]
wheel_path = root / "dist" / wheel_name
observed_sha = hashlib.sha256(wheel_path.read_bytes()).hexdigest()
expected_input_sha = inputs["extension_wheel_sha256"]
expected_manifest_sha = manifest.artifact_sha256[wheel_name]
if not (observed_sha == expected_input_sha == expected_manifest_sha):
    raise RuntimeError("exact Customer certification extension wheel hash mismatch")

# Pure-Python wheel: import exact verified customer extension directly; no inline %pip.
sys.path.insert(0, str(wheel_path))
from fabric_customer_certification_extensions.pipeline_worker import execute_certification_dataset

if not key_vault_url or not control_plane_secret_name or not warehouse_secret_name:
    raise RuntimeError("Key Vault URL and certification secret names must be configured")
os.environ["CONTROL_PLANE_DATABASE_URL"] = credentials.getSecret(key_vault_url, control_plane_secret_name)
os.environ["WAREHOUSE_DATABASE_URL"] = credentials.getSecret(key_vault_url, warehouse_secret_name)
os.environ["CERTIFICATION_PIPELINE_WORKER_CONFIG_PATH"] = str(root / "project/config/certification/pipeline-worker.json")

configs = load_dataset_configs(root / "project/config/datasets")
engine = create_engine(os.environ["CONTROL_PLANE_DATABASE_URL"])
try:
    repository = SqlAlchemyControlPlaneRepository(
        engine,
        domain=manifest.domain,
        domain_git_sha=manifest.bundle.domain_git_sha,
        framework_version=manifest.bundle.framework_version,
        configs=configs,
    )
    request = pipeline_child_request_from_parameters({
        "framework_pipeline_run_id": framework_pipeline_run_id,
        "framework_dataset_run_id": framework_dataset_run_id,
        "dataset_id": dataset_id,
        "run_mode": run_mode,
        "attempt": attempt,
        "effective_config_hash": effective_config_hash,
        "execution_plan_hash": execution_plan_hash,
    })
    outcome = execute_pipeline_child(
        repository=repository,
        request=request,
        executor=execute_certification_dataset,
    )
    print("framework_dataset_run_id =", outcome.dataset_run_id)
    print("framework_status =", outcome.status.value)
finally:
    engine.dispose()
    os.environ.pop("CONTROL_PLANE_DATABASE_URL", None)
    os.environ.pop("WAREHOUSE_DATABASE_URL", None)